In [28]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.IRSwaps.IRSwapQuery import IRSwapQuery

In [34]:
from SDRUtils.data.builder import SDRDataBuilder
from SDRUtils.core.classification import classifications_to_dataframe
from SDRUtils.products.filters import new_sofr_swap_trades 
from SDRUtils.products.usd_swaps import classify_sofr_swap_trade, detect_ust_mms_trades_df
from SDRUtils.packages.curve import detect_curve_trades_df
from SDRUtils.packages.fly import detect_fly_trades_df
from SDRUtils.packages.utils import merge_package_legs_to_one_row

In [32]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)

start = NY_tz.localize(datetime.datetime(2025, 12, 29, 00, 1))
end = NY_tz.localize(datetime.datetime(2025, 12, 29, 20, 00))

raw_df = sdr.grab_sdr_trades(
    start_timestamp=start, end_timestamp=end, agency="CFTC", asset_class="RATES", filter_func=new_sofr_swap_trades,
    # start_timestamp=start, end_timestamp=end, agency="CFTC", asset_class="RATES", filter_func=filter_new_sofr_swaption_trades,
    # start_timestamp=start, end_timestamp=end, agency="CFTC", asset_class="RATES",
)

swaps_mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-RL_BASIC")
curve = swaps_mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

CONCAT...: 100%|██████████| 2/2 [00:00<00:00, 233.28it/s]


In [35]:
from tqdm import tqdm

classifications = []
it = raw_df.iterrows()

it = tqdm(it, total=len(raw_df), desc="Classifying trades", unit="trade")

for idx, row in it:
	trade_id = int(row.get("Dissemination Identifier", idx))
	try:
		classification = classify_sofr_swap_trade(row, trade_id, curve)
		classifications.append(classification)
	except Exception as e:
		continue

classifications_df = classifications_to_dataframe(classifications)
with_pkg_df = merge_package_legs_to_one_row(detect_ust_mms_trades_df(detect_curve_trades_df(detect_fly_trades_df(classifications_df))))

Classifying trades: 100%|██████████| 2121/2121 [00:11<00:00, 188.21trade/s]


In [36]:
with_pkg_df

,trade_id,execution_timestamp,effective_date,expiration_date,product_type,tenor_years,tenor_label,is_forward,forward_start_years,forward_label,...,strike,estimated_pv01,package_type,package_id,package_legs,matched_ust_maturity,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date
0,1572688072000000301,2025-12-29 05:10:11+00:00,2026-01-20,2036-01-20 00:00:00,OIS_SWAP,10.15,10Y,True,0.061111,3W,...,NaN,50205.509059,OUTRIGHT,None,None,False,NaN,NaN,NaN,2036-01-20
1,1572660520000000301,2025-12-29 05:10:11+00:00,2026-01-20,2036-01-20 00:00:00,OIS_SWAP,10.15,10Y,True,0.061111,3W,...,NaN,50205.509059,OUTRIGHT,None,None,False,NaN,NaN,NaN,2036-01-20
2,1563647051000000101,2025-12-29 05:11:56+00:00,2025-12-31,2026-12-28 00:00:00,OIS_SWAP,1.005556,1Y,False,0.005556,spot,...,NaN,29150.839208,OUTRIGHT,None,None,False,NaN,NaN,NaN,2026-12-28
3,1563670963000000101,2025-12-29 05:19:44+00:00,2025-12-31,2026-03-30 00:00:00,OIS_SWAP,0.247222,3M,False,0.005556,spot,...,NaN,4897.953618,OUTRIGHT,None,None,False,NaN,NaN,NaN,2026-03-30
4,1563737164000000101,2025-12-29 05:39:58+00:00,2025-12-31,2045-12-31 00:00:00,OIS_SWAP,20.286111,20Y,False,0.005556,spot,...,NaN,8281.877471,OUTRIGHT,None,None,False,NaN,NaN,NaN,2045-12-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1785,1573666981000000101 / 1573660614000000101,2025-12-29T21:55:48+00:00 / 2025-12-29T21:55:3...,2025-12-31,2035-12-31T00:00:00 / 2055-12-31T00:00:00,OIS_SWAP,10.1444 / 30.4361,10Y / 30Y,False,0.005556,spot,...,NaN,41899.5 / 43204.1,CURVE,CURVE_309,"[1573660614000000101, 1573666981000000101]",False,NaN,NaN,NaN,2035-12-31 / 2055-12-31
1786,1573691775000000101,2025-12-29 21:59:26+00:00,2025-12-31,2032-12-31 00:00:00,OIS_SWAP,7.102778,7Y,False,0.005556,spot,...,NaN,2481.177386,SPREADOVER,SPREADOVER_1573691775000000101,[1573691775000000101],True,91282CPQ8,7-Year,2025-12-31,2032-12-31
1787,1573723141000000201,2025-12-29 21:52:38+00:00,2025-12-31,2055-11-15 00:00:00,OIS_SWAP,30.308333,30Y,False,0.005556,spot,...,0.041224,100055.04316,SPREADOVER,SPREADOVER_1573723141000000201,[1573723141000000201],True,912810UP1,30-Year,2025-11-17,2055-11-15
1788,1573723140000000101,2025-12-29 21:56:25+00:00,2025-12-31,2035-11-15 00:00:00,OIS_SWAP,10.016667,10Y,False,0.005556,spot,...,0.037359,99566.158534,SPREADOVER,SPREADOVER_1573723140000000101,[1573723140000000101],True,91282CPJ4,10-Year,2025-11-17,2035-11-15


In [37]:
# with_pkg_df["estimated_pv01"].sort_values(key=lambda x: float(str(x).split("/")[0]) if type(x) == str else float(x))

# with_pkg_df["pv01_clean"] = with_pkg_df["estimated_pv01"].apply(lambda x: float(str(x).split("/")[0]) if type(x) == str else float(x))
# with_pkg_df.sort_values(by="pv01_clean", ascending=False).head(10)